In [ ]:
# ==========================================
# 這是給您在 VS Code 執行的真實資料抓取程式碼
# ==========================================
import tejapi
import pandas as pd
import numpy as np

# 1. 設定 API Key (請換成您的 Key)
tejapi.ApiConfig.api_key = "i6IHtGDc4v0jZ7lYb8uDK8ZpgcPS3g"
tejapi.ApiConfig.ignoretz = True # 忽略時區

print("正在下載資料...這可能需要一點時間")

# 2. 設定時間範圍 (近10年)
start_date = '2014-01-01'
end_date = '2023-12-31'

# 3. 下載 ESG 資料 (TEJ 資料庫代碼: TWN/AESG)
# 欄位說明: code(代號), date(年月), A_Score(總分), E_Score, S_Score, G_Score
esg_data = tejapi.get('TWN/AESG',
                      mdate={'gte': start_date, 'lte': end_date},
                      opts={'columns': ['coid', 'mdate', 'a_score', 'e_score', 's_score', 'g_score']},
                      paginate=True)

# 整理 ESG 資料 (通常一年一評，我們取年份)
esg_data['year'] = esg_data['mdate'].dt.year
esg_data = esg_data.rename(columns={'coid': 'Company', 'a_score': 'Total_ESG'})

# 4. 下載股價資料並計算年報酬率 (TEJ 資料庫代碼: TWN/APRCD - 調整後股價)
# 為了節省流量，我們只抓每年的年底收盤價來算年報酬
price_data = tejapi.get('TWN/APRCD',
                        mdate={'gte': start_date, 'lte': end_date},
                        opts={'columns': ['coid', 'mdate', 'close_adj']},
                        paginate=True)

# 轉換為年資料 (取每年最後一筆交易)
price_data['year'] = price_data['mdate'].dt.year
yearly_price = price_data.sort_values('mdate').groupby(['coid', 'year']).last().reset_index()

# 計算年報酬率 (今年收盤 - 去年收盤) / 去年收盤
yearly_price['Return'] = yearly_price.groupby('Company')['close_adj'].pct_change() * 100

# 5. 下載公司基本資料 (為了拿 "產業" 當控制變數) (TEJ 代碼: TWN/AIND)
info_data = tejapi.get('TWN/AIND',
                       opts={'columns': ['coid', 'tejinm3_c']}, # tejinm3_c 是 TEJ 子產業名稱
                       paginate=True)
info_data = info_data.rename(columns={'coid': 'Company', 'tejinm3_c': 'Industry'})

# 6. 合併所有資料
# ESG 資料通常是解釋 "下一年度" 或 "當年度" 的績效，這裡我們假設看當年度關聯
df_final = pd.merge(yearly_price, esg_data, on=['Company', 'year'], how='inner')
df_final = pd.merge(df_final, info_data, on='Company', how='left')

# 去除空值 (有些剛上市的公司沒有前一年股價算報酬率)
df_final = df_final.dropna()

print("資料準備完成！")
# df_final.to_csv('tej_esg_analysis.csv') # 可以存起來